## processing semart data to have same structure


In [ ]:
# v1
import pandas as pd
import json
from pathlib import Path
import os

base_path = Path("../SemArt")

# ==== INPUT FILE PATHS (adjust as needed) ====
csv_paths = {
    "train": os.path.join(base_path, "semart_train.csv"),
    "val": os.path.join(base_path, "semart_val.csv"),
    "test": os.path.join(base_path, "semart_test.csv")
}

json_paths = {
    "train": os.path.join(base_path, "explain-paintings", "annotations", "semart_topic_annotated_train.json"),
    "test": os.path.join(base_path, "explain-paintings", "annotations", "semart_topic_annotated_test.json")
}

# ==== OUTPUT FILE PATHS ====
output_paths = {
    "train": "triplets_semart_train.json",
    "val": "triplets_semart_val.json",
    "test": "triplets_semart_test.json"
}


# ==== FUNCTIONS ====
def load_json(path):
    with open(path, "r", encoding="latin1") as f:
        return json.load(f)

def create_triplets(df, annotations):
    """Create triplets from CSV and JSON annotations."""
    img_to_ann = {a["img"]: a for a in annotations["annotations"]}
    triplets = []
    for _, row in df.iterrows():
        img = row["IMAGE_FILE"]
                
        if img not in img_to_ann:
            continue
        ann = img_to_ann[img]

        # Generate triplets for each textual field
        for field in ["description", "content", "form", "context"]:
            text_values = ann.get(field, [])
            if isinstance(text_values, str):
                text_values = [text_values]
            for text in text_values:
                triplets.append({
                    "item1": f"Images/{img}",
                    "item2": text,
                    "link": field
                })
    print(f"Created {len(triplets)} triplets for {len(df)} images.")
    return triplets

train_df = pd.read_csv(csv_paths["train"],  encoding="latin1", sep="\t", dtype=str)
test_df = pd.read_csv(csv_paths["test"],  encoding="latin1", sep="\t", dtype=str)
train_ann = load_json(json_paths["train"])
test_ann = load_json(json_paths["test"])
# Split train into train + val (80/20)
train_df = train_df.sample(frac=1, random_state=42)  # shuffle
val_size = int(0.1 * len(train_df))
val_df = train_df.iloc[:val_size]
train_df = train_df.iloc[val_size:]
# Create triplets
train_triplets = create_triplets(train_df, train_ann)
val_triplets = create_triplets(val_df, train_ann)
test_triplets = create_triplets(test_df, test_ann)
# Save outputs
for split, data in [("train", train_triplets), ("val", val_triplets), ("test", test_triplets)]:
    with open(output_paths[split], "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
print("Triplet JSON files created:")
for k, v in output_paths.items():
    print(f" - {k}: {v}")


In [1]:
# v2
import pandas as pd
import json
from pathlib import Path
import os
import random
from collections import defaultdict

# ================== CONFIG ==================
base_path = Path("../SemArt")
SEED = 42
RATIOS = {"train": 0.8, "val": 0.1, "test_rand": 0.1}  # for the random split of TRAIN pool

csv_paths = {
    "train": os.path.join(base_path, "semart_train.csv"),
    "test":  os.path.join(base_path, "semart_test.csv"),
}

json_paths = {
    "train": os.path.join(base_path, "explain-paintings", "annotations", "semart_topic_annotated_train.json"),
    "test":  os.path.join(base_path, "explain-paintings", "annotations", "semart_topic_annotated_test.json"),
}

# Output files:
# - Original CSV-based test (kept intact)
# - Random split from training pool (80/10/10) with no (image,link) duplicates per split
output_paths = {
    "train": "triplets_semart_train.json",
    "val": "triplets_semart_val.json",
    "test_rand": "triplets_semart_test.json",     # random test from train pool
    "test_csv": "triplets_semart_test_csv.json",  # original CSV-based test set
}

# ================== HELPERS ==================
def load_json(path):
    with open(path, "r", encoding="latin1") as f:
        return json.load(f)

def safe_str(x):
    return "" if pd.isna(x) else str(x)

def first_technique(tech):
    t = safe_str(tech)
    return t.split(",")[0].strip() if t else ""

def metadata_sentences(row):
    """Return {link_key: sentence, ...} from CSV row (skip empties)."""
    author = safe_str(row.AUTHOR)
    timeframe = safe_str(row.TIMEFRAME)
    school = safe_str(row.SCHOOL)
    material = first_technique(row.TECHNIQUE)
    genre = safe_str(row.TYPE)
    md = {
        "author":    f"Artwork by {author}" if author else "",
        "timeframe": f"Artwork painted in {timeframe}" if timeframe else "",
        "school":    f"Artwork from the {school} school" if school else "",
        "material":  f"Artwork made with {material}" if material else "",
        "genre":     f"Artwork of the {genre} genre" if genre else "",
    }
    return {k: v for k, v in md.items() if v}

def build_links_per_image(df, img_to_ann):
    """
    Build a dict: img -> {link_key: [texts...]} including:
      - topic links: description/content/form/context
      - metadata links: author/timeframe/school/material/genre
    """
    links_per_img = defaultdict(lambda: defaultdict(list))
    for row in df.itertuples(index=False):
        img = row.IMAGE_FILE

        # Topic annotations (if present)
        ann = img_to_ann.get(img)
        if ann:
            for field in ["description", "content", "form", "context"]:
                vals = ann.get(field, [])
                if isinstance(vals, str):
                    vals = [vals]
                for t in vals:
                    t = safe_str(t).strip()
                    if t:
                        links_per_img[img][field].append(t)

        # Metadata (from CSV)
        md = metadata_sentences(row)
        for k, sentence in md.items():
            links_per_img[img][k].append(sentence)

    # Drop images with no links
    to_drop = [img for img, d in links_per_img.items() if not any(d.values())]
    for img in to_drop:
        links_per_img.pop(img, None)

    return links_per_img

def merge_annotations(train_ann, test_ann):
    combined = {}
    for src in (train_ann.get("annotations", []), test_ann.get("annotations", [])):
        for a in src:
            if "img" in a:
                combined[a["img"]] = a
    return combined

def weighted_choice(rng, ratios_dict):
    keys = list(ratios_dict.keys())
    weights = [ratios_dict[k] for k in keys]
    # Normalize
    s = sum(weights)
    weights = [w / s for w in weights]
    r = rng.random()
    cum = 0.0
    for k, w in zip(keys, weights):
        cum += w
        if r <= cum:
            return k
    return keys[-1]

def choose_distinct_sets_for_k(rng, ratios_dict, k):
    """
    Choose up to k distinct sets without replacement, roughly weighted by ratios.
    """
    sets = list(ratios_dict.keys())
    # Sample without replacement using weights by repeatedly drawing and removing
    chosen = []
    local_ratios = ratios_dict.copy()
    for _ in range(min(k, len(sets))):
        pick = weighted_choice(rng, local_ratios)
        chosen.append(pick)
        local_ratios = {s: w for s, w in local_ratios.items() if s != pick}
        # Renormalization happens in weighted_choice
    return chosen

def add_triplet(bucket, img, text, link_key):
    bucket.append({"item1": f"Images/{img}", "item2": text, "link": link_key})



In [2]:
# ================== LOAD DATA ==================
random.seed(SEED)

train_df = pd.read_csv(csv_paths["train"], encoding="latin1", sep="\t", dtype=str)
test_df  = pd.read_csv(csv_paths["test"],  encoding="latin1", sep="\t", dtype=str)

train_ann = load_json(json_paths["train"])
test_ann  = load_json(json_paths["test"])
img_to_ann_all = merge_annotations(train_ann, test_ann)

# Separate mapping for train-pool and for csv-test
img_to_ann_train = {k: v for k, v in img_to_ann_all.items() if k in set(train_df["IMAGE_FILE"])}
img_to_ann_test  = {k: v for k, v in img_to_ann_all.items() if k in set(test_df["IMAGE_FILE"])}

# ================== 1) ORIGINAL CSV-BASED TEST ==================
links_per_img_test_csv = build_links_per_image(test_df, img_to_ann_test)
test_csv_triplets = []
for img, linkmap in links_per_img_test_csv.items():
    for link_key, texts in linkmap.items():
        for t in texts:
            add_triplet(test_csv_triplets, img, t, link_key)

with open(output_paths["test_csv"], "w", encoding="utf-8") as f:
    json.dump(test_csv_triplets, f, indent=2, ensure_ascii=False)
print(f"test_csv: {len(test_csv_triplets)} triplets -> {output_paths['test_csv']}")

# ================== 2) RANDOM 80/10/10 SPLIT FROM TRAIN POOL ==================
links_per_img_train = build_links_per_image(train_df, img_to_ann_train)

splits = {"train": [], "val": [], "test_rand": []}

# Track (img, link) presence per split to guarantee no duplicates
seen = {s: set() for s in splits.keys()}

rng = random.Random(SEED)

for img, linkmap in links_per_img_train.items():
    for link_key, texts in linkmap.items():
        if not texts:
            continue
        texts = [t for t in texts if t and str(t).strip()]
        if not texts:
            continue

        rng.shuffle(texts)

        if len(texts) >= 3:
            # Ensure one in each split (train, val, test_rand). Extra texts dropped to avoid duplicates.
            target_sets = ["train", "val", "test_rand"]
        elif len(texts) == 2:
            # Pick two distinct sets by ratios
            target_sets = choose_distinct_sets_for_k(rng, RATIOS, 2)
        else:
            # Single text -> pick one set by ratios
            target_sets = [weighted_choice(rng, RATIOS)]

        # Assign at most one (img, link) to each chosen set
        for s, t in zip(target_sets, texts):
            key = (img, link_key)
            if key in seen[s]:
                # Already have (img,link) in this split; skip to maintain uniqueness
                continue
            add_triplet(splits[s], img, t, link_key)
            seen[s].add(key)

# ================== SAVE RANDOM SPLIT ==================
for split_name in ["train", "val", "test_rand"]:
    with open(output_paths[split_name], "w", encoding="utf-8") as f:
        json.dump(splits[split_name], f, indent=2, ensure_ascii=False)
    print(f"{split_name}: {len(splits[split_name])} triplets -> {output_paths[split_name]}")

print("Done. Files written:")
for k, v in output_paths.items():
    print(f" - {k}: {v}")


test_csv: 9914 triplets -> triplets_semart_test_csv.json
train: 115826 triplets -> triplets_semart_train.json
val: 25703 triplets -> triplets_semart_val.json
test_rand: 25615 triplets -> triplets_semart_test.json
Done. Files written:
 - train: triplets_semart_train.json
 - val: triplets_semart_val.json
 - test_rand: triplets_semart_test.json
 - test_csv: triplets_semart_test_csv.json


In [ ]:
# v3
import pandas as pd
import json
from pathlib import Path
import os

base_path = Path("../SemArt")

# ==== INPUT FILE PATHS (adjust as needed) ====
csv_paths = {
    "train": os.path.join(base_path, "semart_train.csv"),
    "val": os.path.join(base_path, "semart_val.csv"),  # not used directly; we split train below
    "test": os.path.join(base_path, "semart_test.csv")
}

json_paths = {
    "train": os.path.join(base_path, "explain-paintings", "annotations", "semart_topic_annotated_train.json"),
    "test": os.path.join(base_path, "explain-paintings", "annotations", "semart_topic_annotated_test.json")
}

# ==== OUTPUT FILE PATHS ====
output_paths = {
    "train": "triplets_semart_train.json",
    "val": "triplets_semart_val.json",
    "test": "triplets_semart_test.json"
}



In [2]:
# ==== HELPERS ====
def load_json(path):
    with open(path, "r", encoding="latin1") as f:
        return json.load(f)

def safe_str(x):
    return "" if pd.isna(x) else str(x)

def first_technique(tech):
    # Take the first technique before a comma, if present
    t = safe_str(tech)
    return t.split(",")[0].strip() if t else ""

def metadata_sentences(row):
    """Build a dict of {link: sentence} for metadata fields present in the CSV row."""
    author = safe_str(row.AUTHOR)
    timeframe = safe_str(row.TIMEFRAME)
    school = safe_str(row.SCHOOL)
    material = first_technique(row.TECHNIQUE)
    genre = safe_str(row.TYPE)

    md = {
        "author": f"Artwork by {author}" if author else "",
        "timeframe": f"Artwork painted in {timeframe}" if timeframe else "",
        "school": f"Artwork from the {school} school" if school else "",
        "material": f"Artwork made with {material}" if material else "",
        "genre": f"Artwork of the {genre} genre" if genre else "",
    }
    # Drop empties
    return {k: v for k, v in md.items() if v}

def create_triplets(df, annotations):
    """
    Create triplets from CSV rows + JSON annotations.
    Adds both topic annotations (description/content/form/context)
    and metadata (author/timeframe/school/material/genre) as separate links.
    """
    img_to_ann = {a["img"]: a for a in annotations.get("annotations", [])}
    triplets = []

    # Iterate rows once; avoid expensive per-row filtering
    for row in df.itertuples(index=False):
        img = row.IMAGE_FILE

        # 1) Topic annotation triplets (if available)
        ann = img_to_ann.get(img)
        if ann:
            for field in ["description", "content", "form", "context"]:
                text_values = ann.get(field, [])
                if isinstance(text_values, str):
                    text_values = [text_values]
                for text in text_values:
                    if text and str(text).strip():
                        triplets.append({
                            "item1": f"Images/{img}",
                            "item2": str(text),
                            "link": field
                        })

        # 2) Metadata triplets (always, using CSV columns if present)
        md = metadata_sentences(row)
        for link_key, sentence in md.items():
            triplets.append({
                "item1": f"Images/{img}",
                "item2": sentence,
                "link": link_key
            })

    print(f"Created {len(triplets)} triplets for {len(df)} images.")
    return triplets

# ==== LOAD DATA ====
train_df = pd.read_csv(csv_paths["train"],  encoding="latin1", sep="\t", dtype=str)
test_df  = pd.read_csv(csv_paths["test"],   encoding="latin1", sep="\t", dtype=str)

train_ann = load_json(json_paths["train"])
test_ann  = load_json(json_paths["test"])



In [3]:
# ==== SPLIT TRAIN INTO TRAIN/VAL (90/10 here) ====
train_df = train_df.sample(frac=1, random_state=42)  # shuffle
val_size = int(0.1 * len(train_df))
val_df = train_df.iloc[:val_size].copy()
train_df = train_df.iloc[val_size:].copy()

# ==== CREATE TRIPLETS ====
train_triplets = create_triplets(train_df, train_ann)
val_triplets   = create_triplets(val_df,   train_ann)
test_triplets  = create_triplets(test_df,  test_ann)

# ==== SAVE ====
for split, data in [("train", train_triplets), ("val", val_triplets), ("test", test_triplets)]:
    with open(output_paths[split], "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

print("Triplet JSON files created:")
for k, v in output_paths.items():
    print(f" - {k}: {v}")


Created 167494 triplets for 17320 images.
Created 18779 triplets for 1924 images.
Created 9914 triplets for 1069 images.
Triplet JSON files created:
 - train: triplets_semart_train.json
 - val: triplets_semart_val.json
 - test: triplets_semart_test.json
